# Week 2 conditions — latent-agent-cost

Latent condition + ablations for *Tokens Are Not Dollars* (Qwen2.5-1.5B-Instruct, GSM8K + ProsQA, 500 tasks x 3 seeds each):

- `latent` — training-free Coconut-style hidden-state feedback, 16 latent steps, then explicit answer decoding
- `latent_zero` — zero-latent ablation (same schedule, states zeroed)
- `latent_perturb` — perturbation ablation (Gaussian noise, scale 0.1, added to the latent states)
- `pause_token` — pause-token control (16 filler-token positions, then explicit CoT)

Corrected Week-1 protocol throughout: max 768 new tokens, the shared prompt with its closing `#### <answer>` instruction, unchanged verifiers.

**Before running:** notebook Settings -> **Accelerator = GPU** (T4 or P100) and **Internet = ON** (needed for the git clone, the HF dataset downloads, and the model download).

**Runtime warning:** this is 8 run cells of 1,500 attempts each (vs. 2 cells in Week 1, which took ~7.5 GPU-hours). Expect roughly 25–30 GPU-hours in total — more than one Kaggle session and close to the 30 h/week free quota. Run the cells in stages across sessions if needed: attempts are appended and flushed to each cell's `.attempts.jsonl` after every attempt, so a cut-off keeps completed attempts (summary tables are written at each cell's end). Use **Save Version -> Save & Run All (Commit)** per stage so outputs persist (see `RUNBOOK.md`).

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

**latent / GSM8K** — Latent condition — 16 hidden-state feedback steps, then explicit answer decoding.

In [ ]:
!python scripts/run_baseline.py --dataset gsm8k --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent --latent-steps 16 --perturb-scale 0.1 --out results/week2_latent_gsm8k --label WEEK2

**latent / ProsQA** — Latent condition — 16 hidden-state feedback steps, then explicit answer decoding.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent --latent-steps 16 --perturb-scale 0.1 --out results/week2_latent_prosqa --label WEEK2

**latent_zero / GSM8K** — Zero-latent ablation — same 16-step schedule, fed-back states replaced with zeros (compute-matched; tests whether the states carry task information).

In [ ]:
!python scripts/run_baseline.py --dataset gsm8k --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent_zero --latent-steps 16 --perturb-scale 0.1 --out results/week2_zero_gsm8k --label WEEK2

**latent_zero / ProsQA** — Zero-latent ablation — same 16-step schedule, fed-back states replaced with zeros (compute-matched; tests whether the states carry task information).

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent_zero --latent-steps 16 --perturb-scale 0.1 --out results/week2_zero_prosqa --label WEEK2

**latent_perturb / GSM8K** — Perturbation ablation — Gaussian noise (scale 0.1) added to the fed-back states at every step.

In [ ]:
!python scripts/run_baseline.py --dataset gsm8k --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent_perturb --latent-steps 16 --perturb-scale 0.1 --out results/week2_perturb_gsm8k --label WEEK2

**latent_perturb / ProsQA** — Perturbation ablation — Gaussian noise (scale 0.1) added to the fed-back states at every step.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent_perturb --latent-steps 16 --perturb-scale 0.1 --out results/week2_perturb_prosqa --label WEEK2

**pause_token / GSM8K** — Pause-token control — 16 filler-token positions processed before explicit CoT: the latent budget spent on visible filler instead of latent states.

In [ ]:
!python scripts/run_baseline.py --dataset gsm8k --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition pause_token --latent-steps 16 --perturb-scale 0.1 --out results/week2_pause_gsm8k --label WEEK2

**pause_token / ProsQA** — Pause-token control — 16 filler-token positions processed before explicit CoT: the latent budget spent on visible filler instead of latent states.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 1 2 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition pause_token --latent-steps 16 --perturb-scale 0.1 --out results/week2_pause_prosqa --label WEEK2

Show the resulting five-metric tables.

In [ ]:
!ls -la results/
!cat results/week2_latent_gsm8k.md
!cat results/week2_latent_prosqa.md
!cat results/week2_zero_gsm8k.md
!cat results/week2_zero_prosqa.md
!cat results/week2_perturb_gsm8k.md
!cat results/week2_perturb_prosqa.md
!cat results/week2_pause_gsm8k.md
!cat results/week2_pause_prosqa.md